# 02 · First recon
*Your first look at the raw feeds, before anyone cleans anything.*

Agent Coulson's team has dropped a pile of field reports. In this notebook you generate the seed data into a volume, read it with Spark, summarise it with DataFrames and with SQL, and then hunt for the defects the generator planted on purpose.

| Concept | Where you meet it |
|---|---|
| A DataFrame is a lazy plan; actions run it | the next section, Steps 2 and 3 |
| Schema inference when reading JSON | Step 2 |
| Transformations, and why `groupBy` is expensive (narrow vs wide) | Step 4 |
| SQL and DataFrames are the same engine (temp views) | Step 5 |
| Reading data critically: defects and duplicates | Steps 6 and 7 |

Deep dive: [docs/02-first-recon.md](../../docs/02-first-recon.md)

**Needs:** chapter 01 ran (the volume `shield.landing.staging`), and this repo is cloned as a **Git folder** so `shield_data` can be imported (chapter 00 checks that). Serverless compute is the simplest choice.
The **Expected** numbers assume the dataset this notebook generates (10,000 incidents, seed 42, 5 % dirty); other data gives other numbers and a ❌, which is not a bug. They are expectations until you have run the notebook in a workspace.

## How Spark reads your question

**The idea:** a **DataFrame** is a table with a schema that is spread over many machines. When you write `filter`, `select` or `groupBy` you are not computing anything, you are **adding a step to a plan**. Only an **action** (`display`, `count`, `write`) makes Spark run the plan, and it can then optimise the whole plan at once. This is how Spark works in general, not something Databricks added.

```
  read ──▶ filter / select ──▶ groupBy ──▶ orderBy ──▶ display(...)
  (plan)   narrow: each        wide: rows with the      ACTION: now the whole
           partition alone     same key must meet       plan actually runs
                               = SHUFFLE (Exchange)
```

**How this notebook implements it**

* Step 2 builds a DataFrame with `spark.read.json`. To learn the column types Spark has to look at the data once, so even this line does a little work.
* Step 3 triggers real work with `count()` and `display`.
* Step 4 groups by incident type and severity. `filter` and `select` work on one partition at a time (**narrow**), `groupBy` and `join` move rows between machines (**wide**, a *shuffle*). You will find it in the plan as `Exchange` in mini-project 2 of the doc.
* Step 5 repeats Step 4 in SQL through a temporary view: both languages produce the same plan.

**Compute in one table:** *serverless* starts in seconds, is managed for you and bills per use; a *classic cluster* lets you pick size and runtime and you stop it yourself. This course assumes serverless.

**Why it helps:** knowing what is lazy and what is a shuffle explains most "why is this slow?" questions later.

**Not covered yet:** Auto Loader instead of `spark.read.json`, which avoids re-inferring the schema from the data on every read (chapter 03), and Delta tables (chapter 04).

## Setup

In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "src")))
from pyspark.sql import functions as F

from shield_data.generate import generate

STAGING = Path("/Volumes/shield/landing/staging")
SEVERITIES = ["low", "medium", "high", "critical"]


def show_checks(checks):
    """Show (check, result, expected) rows with a ✅ / ❌ for whether result matches expected."""
    fmt = lambda v: f"{v:,}" if isinstance(v, int) and not isinstance(v, bool) else str(v)
    rows = [(what, fmt(got), fmt(want), "✅" if got == want else "❌") for what, got, want in checks]
    display(spark.createDataFrame(rows, "check string, result string, expected string, ok string"))


assert STAGING.is_dir(), f"{STAGING} not found. Run notebook 01 first (it creates the volume)."

## 1 · Generate and read

### Step 1 · Generate the seed data
**Goal:** write the synthetic feeds into the staging volume. Everything in this course uses this data and nothing else.

| Input | Value |
|---|---|
| `generate(...)` | 10,000 incidents, 20,000 sensor pings, `seed=42`, `dirty=0.05`, `batches=5` |

**Expected:** the incident feed has 10,250 rows (10,000 incidents plus 250 replayed duplicates) split over five files `part-0000.json` to `part-0004.json`. The generator also writes 10 locations, 50 agents, 120 threat assessments and 20,000 sensor pings.
The five files are on purpose: they "arrive" one at a time in chapter 03.

Safe to re-run: the same seed writes the same files over the old ones. Only the timestamps of the planted "future" defects move with the day you run it. Chapter 03 works on copies, so regenerating does not change files it already ingested.

In [ ]:
counts = generate(STAGING, incidents=10_000, pings=20_000, seed=42, dirty=0.05, batches=5)

show_checks([
    ("incident_reports rows", counts["incident_reports"], 10_250),
    ("sensor_pings rows", counts["sensor_pings"], 20_000),
    ("locations rows", counts["locations"], 10),
    ("agents rows", counts["agents"], 50),
    ("threat_assessments rows", counts["threat_assessments"], 120),
    ("incident files", sorted(p.name for p in (STAGING / "incident_reports").glob("*")), [f"part-{i:04d}.json" for i in range(5)]),
])

### Step 2 · Read the JSON: the schema
**Goal:** create the DataFrame and see the schema Spark inferred. Nothing has been computed on the rows yet.

**Expected:** nine columns. Spark sorts inferred JSON fields alphabetically, so the order differs from the files. `casualties` is a number (`long`), everything else is `string`, including `reported_at`: a timestamp written as text stays text until you parse it (chapter 05). `source_channel` exists although only the last files have it, because Spark looks at all files.

```
root
 |-- agent_id: string (nullable = true)
 |-- casualties: long (nullable = true)
 |-- incident_id: string (nullable = true)
 |-- incident_type: string (nullable = true)
 |-- location_id: string (nullable = true)
 |-- reported_at: string (nullable = true)
 |-- severity: string (nullable = true)
 |-- source_channel: string (nullable = true)
 |-- summary: string (nullable = true)
```

Safe to re-run.

In [ ]:
incidents = spark.read.json(f"{STAGING}/incident_reports")
incidents.printSchema()

column_types = dict(incidents.dtypes)
show_checks([
    ("columns", incidents.columns, sorted(["incident_id", "reported_at", "agent_id", "location_id", "incident_type", "severity", "casualties", "summary", "source_channel"])),
    ("casualties type", column_types["casualties"], "bigint"),
    ("reported_at type", column_types["reported_at"], "string"),
])

### Step 3 · An action runs the plan
**Goal:** see the difference between a plan and a result. `count()` and `display` are actions: they read the files.

**Expected:** 10,250 rows in total. The table shows only the first 10 rows (`limit(10)`), because the full table is 10,250 rows long.

For example (row order is not guaranteed):

| incident_id | reported_at | agent_id | location_id | incident_type | severity | casualties |
|---|---|---|---|---|---|---|
| I0000001 | 2025-01-23T17:06:13Z | A0038 | L009 | surveillance | medium | 2 |
| I0000002 | 2025-07-15T06:54:18Z | A0034 | L003 | surveillance | critical | 1 |
| I0000003 | 2025-02-01T02:41:34Z | A0012 | L002 | smuggling | low | 1 |

Safe to re-run: reading does not change anything. Run it twice and time `count()`: the second run is no faster, because Spark caches nothing unless you ask.

In [ ]:
display(incidents.limit(10))

show_checks([("incident rows", incidents.count(), 10_250)])

## 2 · Summarise it

### Step 4 · Transformations: group and aggregate
**Goal:** count incidents and sum casualties per incident type and severity. This is a wide transformation: Spark shuffles rows so that all rows of one group meet.

**Expected:** 30 groups (6 incident types × 5 severity values; the fifth value, `catastrophic`, is not a real severity, you will find out in Step 6). The largest group is `surveillance` / `low` with 895 incidents and 501 casualties. This counts raw rows, so duplicates and defects are included on purpose.

The top of the table:

| incident_type | severity | n | casualties |
|---|---|---|---|
| surveillance | low | 895 | 501 |
| enhanced_activity | low | 876 | 467 |
| breach | low | 837 | 454 |

Safe to re-run.

In [ ]:
by_type = (
    incidents.groupBy("incident_type", "severity")
    .agg(F.count("*").alias("n"), F.sum("casualties").alias("casualties"))
    .orderBy(F.desc("n"))
)
display(by_type)

largest_group = by_type.first()
show_checks([
    ("groups", by_type.count(), 30),
    ("largest group", (largest_group["incident_type"], largest_group["severity"], largest_group["n"], largest_group["casualties"]), ("surveillance", "low", 895, 501)),
])

### Step 5 · The same thing in SQL
**Goal:** show that SQL and DataFrames are one engine. A **temp view** gives a DataFrame a name that SQL can use.

**Expected:** exactly the table of Step 4. The check compares the two results in both directions.

The view `incidents_tmp` exists only in this notebook session and is replaced when you re-run, so the cell is safe to repeat.

In [ ]:
incidents.createOrReplaceTempView("incidents_tmp")
by_type_sql = spark.sql(
    """
    SELECT incident_type, severity, COUNT(*) AS n, SUM(casualties) AS casualties
    FROM incidents_tmp
    GROUP BY incident_type, severity
    ORDER BY n DESC
    """
)
display(by_type_sql)

show_checks([
    ("rows", by_type_sql.count(), 30),
    ("rows only in the DataFrame result", by_type.exceptAll(by_type_sql).count(), 0),
    ("rows only in the SQL result", by_type_sql.exceptAll(by_type).count(), 0),
])

## 3 · Spot the mess

The generator injected defects (5 % of the incidents get exactly one) and replayed some deliveries. Find them now, by hand. Chapter 06 turns this into rules.

### Step 6 · Defective values
**Goal:** count the impossible values: a missing agent, a severity that does not exist, negative casualties, a timestamp that is not a timestamp, a timestamp in the future.

**Expected** (rows, duplicates included):

| check | expected |
|---|---|
| missing agent | 96 |
| severity not in low / medium / high / critical | 110 |
| negative casualties | 96 |
| unparsable `reported_at` | 99 |
| `reported_at` in the future | 110 |

The severity breakdown shows five values: `low` 5,036, `medium` 3,123, `high` 1,512, `critical` 469 and the intruder `catastrophic` 110.
The "future" count holds for data generated recently: the generator plants those timestamps 30 to 400 days after the day it ran.

Safe to re-run.

In [ ]:
reported_ts = F.expr("try_to_timestamp(reported_at, \"yyyy-MM-dd'T'HH:mm:ss'Z'\")")
defects = incidents.withColumn("reported_ts", reported_ts).agg(
    F.sum(F.when(F.col("agent_id").isNull(), 1).otherwise(0)).alias("missing_agent"),
    F.sum(F.when(~F.col("severity").isin(SEVERITIES), 1).otherwise(0)).alias("invalid_severity"),
    F.sum(F.when(F.col("casualties") < 0, 1).otherwise(0)).alias("negative_casualties"),
    F.sum(F.when(F.col("reported_ts").isNull(), 1).otherwise(0)).alias("unparsable_timestamp"),
    F.sum(F.when(F.col("reported_ts") > F.current_timestamp(), 1).otherwise(0)).alias("future_timestamp"),
).first()

display(incidents.groupBy("severity").count().orderBy(F.desc("count")))
show_checks([
    ("missing agent", defects["missing_agent"], 96),
    ("invalid severity", defects["invalid_severity"], 110),
    ("negative casualties", defects["negative_casualties"], 96),
    ("unparsable timestamp", defects["unparsable_timestamp"], 99),
    ("timestamp in the future", defects["future_timestamp"], 110),
])

### Step 7 · Duplicates
**Goal:** find incidents that were delivered twice. A replayed delivery re-sends a row that already exists.

**Expected:** 10,250 rows but only 10,000 distinct `incident_id` values, so 250 ids appear twice. The table lists 5 of the 250 (`limit(5)`, to keep it readable).

Safe to re-run.

In [ ]:
duplicated_ids = incidents.groupBy("incident_id").count().filter("count > 1")
display(duplicated_ids.limit(5))

show_checks([
    ("rows", incidents.count(), 10_250),
    ("distinct incident_id", incidents.select("incident_id").distinct().count(), 10_000),
    ("incident_ids that appear twice", duplicated_ids.count(), 250),
])

## 🧹 Clean up

Nothing to delete. Step 1 left the seed data in `shield.landing.staging`, and chapters 03, 09, 10 and 12 read it. The temp view `incidents_tmp` disappears with your notebook session.

## 🎯 Recap

| Idea | In this notebook |
|---|---|
| A DataFrame is a lazy plan | Step 2 builds it, Step 3's `count()` and `display` run it |
| `groupBy` is a wide transformation | Step 4 (look for `Exchange` in `explain()`) |
| SQL and DataFrames are one engine | Step 5 gives identical results |
| Raw data is messy | Steps 6 and 7 found defects and duplicates by hand |

Nothing was cleaned or stored: bronze must keep all of it. Chapter 03 lands these files in Delta tables without touching them.

Try-it-yourself mini-projects: [docs/02-first-recon.md](../../docs/02-first-recon.md#try-it-yourself)

> 💡 Next: chapter 03 ("Coulson ingests") delivers these files one by one and loads them incrementally with Auto Loader.